# AlphaBuilder v4.1 — Etapa 6: Self-Play Iterativo Curricular (`Phys-UViT 3D`, Kaggle `2 x Tesla T4`)

Notebook pronto para executar a **Etapa 6 v4.1 (MCTS-DAG + Macro-Ações SWAP/STOP + Ciclo de Vida Corrigido + FEM real + Replay Misto)** assim que houver orçamento de GPU no Kaggle:

- **Currículo Progressivo em 4 Tiers**: Cantilever Base (`S1–S6` estritamente excluídos), Apoios Parciais/Dupla Aba/Cargas 3D/Bi-carga, Pontes MBB/Suporte em L/Janelas Proibidas (`domain_mask`), e Multi-Resolução (`48x32x8`, `64x24x8`, `48x24x12`) sem zero-padding.
- **Motor MCTS-DAG v4.1 + op_head**: FPU relativo ao pai, remoção restrita à superfície + veto BFS de desconexão, crescimento bidirecional na Fase 1, macro-ações compostas com `SWAP` conservador de massa e `STOP` voluntário com reuso de subárvore, e cabeça global `op_head` (`[ADD, REMOVE, STOP]`) inicializada a zero compatível com checkpoints Stage 5.
- **Ciclo de Vida Protegido**: *Grace Period* pós-conexão (5 passos), lotes adaptativos (`16 -> 8 -> 4 -> 2`), parada por paciência sobre `R_total` + hash de ciclos + retropropagação protegida pelo pico (`best-of-episode`).
- **Avaliação Determinística (MCTS Puro)** a cada geração: Suíte Canônica **`S1–S6`** + Suíte OOD **`G1–G4`**.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

print("=" * 64)
print("AlphaBuilder v4.0 - Stage 6 Self-Play Pipeline (Kaggle 2x T4)")
print("=" * 64)
print(f"CUDA available: {torch.cuda.is_available()} | GPU count: {torch.cuda.device_count()}")
for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    print(f"  GPU {idx}: {torch.cuda.get_device_name(idx)} ({props.total_memory / 1024**3:.1f} GB)")


In [ ]:
# === Configuracao da Campanha Stage 6 ===
MODE = "smoke"              # Altere para "full" para rodar 5 geracoes (~64 eps/gen)
START_GEN = 1
END_GEN = 1 if MODE == "smoke" else 5
EPISODES_PER_GEN = 8 if MODE == "smoke" else 64
TRAIN_EPOCHS = 2 if MODE == "smoke" else 6
NUM_SIMULATIONS = 64 if MODE == "smoke" else 160
MAX_PHASE2_STEPS = 32 if MODE == "smoke" else 64
MAX_WALL_HOURS = 1.0 if MODE == "smoke" else 10.5
LEARNING_RATE = 5e-5
SELFPLAY_FRACTION = 0.60

REPO_URL = "https://github.com/gustavomello9600/alphabuild.git"
REPO_DIR = Path("/tmp/alphabuild")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyamg"], check=True)
print("Active commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
# === Descoberta Automatica de Warmup DB, Split e Checkpoint Inicial (Gen 0 ou Retomada) ===
import shutil
import time

search_roots = [Path("/kaggle/input"), REPO_DIR / "checkpoints", REPO_DIR / "data" / "compiled", Path("/kaggle/working")]

def _find_first(patterns):
    for pat in patterns:
        for root in search_roots:
            if root.exists():
                matches = sorted(root.rglob(pat))
                if matches:
                    return matches[0]
    return None

src_db = _find_first(["warmup_v3_3_kaggle.db"])
split_path = _find_first(["split_v3_3.json"])
init_ckpt = _find_first([f"phys_uvit_selfplay_gen{START_GEN - 1}.pt", "phys_uvit_v4_stage5_best.pt", "best_model.pt"])

if src_db is None:
    raise FileNotFoundError("warmup_v3_3_kaggle.db nao encontrado em /kaggle/input!")
if init_ckpt is None:
    raise FileNotFoundError("Checkpoint inicial Gen 0 nao encontrado em /kaggle/input!")

LOCAL_DB_PATH = Path("/tmp/warmup_v3_3_kaggle.db")
if not LOCAL_DB_PATH.exists() or LOCAL_DB_PATH.stat().st_size != src_db.stat().st_size:
    t0 = time.perf_counter()
    print(f"Copying {src_db} -> {LOCAL_DB_PATH}...")
    shutil.copyfile(src_db, LOCAL_DB_PATH)
    print(f"Copied in {time.perf_counter() - t0:.1f}s")

# Descoberta opcional de DBs de geracoes anteriores caso seja Part 2
prior_dbs = sorted(str(p) for p in Path("/kaggle/input").rglob("selfplay_gen*.db")) if Path("/kaggle/input").exists() else []

print(f"Warmup DB : {LOCAL_DB_PATH}")
print(f"Split JSON: {split_path}")
print(f"Init Ckpt : {init_ckpt}")
print(f"Prior SP  : {prior_dbs}")


In [ ]:
# === Execucao do Orquestrador Iterativo Stage 6 ===
WORK_DIR = Path("/kaggle/working/stage6_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    "scripts/run_selfplay_pipeline.py",
    "--mode", MODE,
    "--init-checkpoint", str(init_ckpt),
    "--warmup-db", str(LOCAL_DB_PATH),
    "--work-dir", str(WORK_DIR),
    "--start-gen", str(START_GEN),
    "--end-gen", str(END_GEN),
    "--episodes-per-gen", str(EPISODES_PER_GEN),
    "--train-epochs", str(TRAIN_EPOCHS),
    "--lr", str(LEARNING_RATE),
    "--selfplay-fraction", str(SELFPLAY_FRACTION),
    "--num-simulations", str(NUM_SIMULATIONS),
    "--max-phase2-steps", str(MAX_PHASE2_STEPS),
    "--fem-backend", "fast_amg",
    "--max-wall-hours", str(MAX_WALL_HOURS),
]
if split_path is not None:
    cmd.extend(["--split-json", str(split_path)])
if prior_dbs:
    cmd.extend(["--prior-selfplay-dbs", ",".join(prior_dbs)])

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# === Resumo da Evolucao, Graficos & Exportacao ===
import json
import shutil

hist_file = WORK_DIR / "selfplay_evolution_history.json"
if hist_file.exists():
    history = json.loads(hist_file.read_text(encoding="utf-8"))
    for entry in history:
        g = entry["generation"]
        c30 = entry.get("eval_canonical_30x8") or {}
        ext = entry.get("eval_canonical_extended") or {}
        ood = entry.get("eval_ood_g1_g4") or {}
        gs = entry.get("generation_stats") or {}
        print(
            f"Gen {g} ({entry['cumulative_gpu_hours']:.2f} GPU-h): "
            f"S1-S6(30x8) Score={c30.get('mean_score', 0):+.3f} (Best={gs.get('mean_best_score', 0):+.3f}), "
            f"C={c30.get('mean_compliance')}, logC_MAE={c30.get('log_c_ood_mae')}, "
            f"REM%={gs.get('p2_rem_fraction', 0)*100:.1f}%, STOP%={gs.get('stop_fraction', 0)*100:.1f}%, "
            f"Reuse={gs.get('mean_tree_reuse_hits', 0):.1f}, DAG={gs.get('total_dag_transpositions', 0)} | "
            f"ExtScore={ext.get('mean_score', '---')} | OOD={ood.get('mean_score', '---')}"
        )

    try:
        import matplotlib.pyplot as plt
        gens = [h["generation"] for h in history]
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
        axes[0].plot(gens, [h["eval_canonical_30x8"]["mean_score"] for h in history], "o-", label="S1-S6 (30x8)")
        if all(h.get("eval_canonical_extended") for h in history):
            axes[0].plot(gens, [h["eval_canonical_extended"]["mean_score"] for h in history], "s-", label="S1-S6 (64 steps)")
        if all(h.get("eval_ood_g1_g4") for h in history):
            axes[0].plot(gens, [h["eval_ood_g1_g4"]["mean_score"] for h in history], "^--", label="G1-G4 OOD")
        axes[0].set_title("Score v3.3 por Geracao")
        axes[0].set_xlabel("Geracao Self-Play")
        axes[0].legend(); axes[0].grid(True, alpha=0.3)

        axes[1].plot(gens, [h["eval_canonical_30x8"]["log_c_ood_mae"] for h in history], "o-", label="log10(C) OOD MAE")
        axes[1].plot(gens, [h["eval_canonical_30x8"]["value_ood_mae"] for h in history], "s-", label="Value OOD MAE")
        axes[1].set_title("Calibracao OOD")
        axes[1].set_xlabel("Geracao Self-Play")
        axes[1].legend(); axes[1].grid(True, alpha=0.3)

        axes[2].plot(gens, [h["generation_stats"]["p2_rem_fraction"] * 100 for h in history], "o-", label="REMOVE% (Fase 2)")
        axes[2].plot(gens, [h["eval_canonical_30x8"]["mean_hollow_xy"] * 100 for h in history], "s-", label="Hollow% (S1-S6)")
        axes[2].set_title("Escavacao & Trelica")
        axes[2].set_xlabel("Geracao Self-Play")
        axes[2].legend(); axes[2].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig("/kaggle/working/stage6_evolution.png", dpi=150)
        plt.show()
    except Exception as exc:
        print("Plot skipped:", exc)

    shutil.copy2(hist_file, Path("/kaggle/working/selfplay_evolution_history.json"))
for pt in sorted((WORK_DIR / "checkpoints").glob("*.pt")):
    shutil.copy2(pt, Path("/kaggle/working") / pt.name)
    print(f"Exported: /kaggle/working/{pt.name} ({pt.stat().st_size / 1e6:.2f} MB)")
